In [17]:
import pandas as pd

path = "/content/drive/MyDrive/DWDM_Project/data/raw/samsung_clean_dataset.csv"
df = pd.read_csv(path)

df.head()

,Product,Category,Rating,Review,Platform,Date
0,Samsung Buds Live,Accessories,5,worth it house save decide although bag fight ...,Flipkart,2023-01-01
1,Samsung Galaxy M14,Smartphone,2,waste of money issue display flickering,Flipkart,2023-01-01
2,Samsung Galaxy M53,Smartphone,5,excellent game threat see head cover task see,Flipkart,2023-01-01
3,Samsung Galaxy Tab A8,Smartphone,4,good,Amazon,2023-01-01
4,Samsung Charger 25W,Accessories,1,average join hotel daughter water,Flipkart,2023-01-01


In [83]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

def random_date():
    start = datetime(2022, 1, 1)
    end = datetime(2025, 1, 1)
    return start + timedelta(days=random.randint(0, (end - start).days))

df["Date"] = [random_date() for _ in range(len(df))]

In [84]:
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

df["Sentiment_score"] = df["Review"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [85]:
print(df[['Review', 'Sentiment_score']].head())

                                              Review  Sentiment_score
0  worth it house save decide although bag fight ...           0.5859
1            waste of money issue display flickering          -0.4215
2      excellent game threat see head cover task see           0.0772
3                                               good           0.4404
4                  average join hotel daughter water           0.2960


In [86]:
df.head()

,Product,Category,Rating,Review,Platform,Date,Issue,RSI,Sentiment_score
0,Samsung Buds Live,Accessories,5,worth it house save decide although bag fight ...,Flipkart,2024-12-14,none,0.0600,0.5859
1,Samsung Galaxy M14,Smartphone,2,waste of money issue display flickering,Flipkart,2022-07-04,display,0.6536,-0.4215
2,Samsung Galaxy M53,Smartphone,5,excellent game threat see head cover task see,Flipkart,2022-07-21,none,0.0600,0.0772
3,Samsung Galaxy Tab A8,Smartphone,4,good,Amazon,2024-03-13,none,0.1350,0.4404
4,Samsung Charger 25W,Accessories,1,average join hotel daughter water,Flipkart,2024-11-13,heating,0.5600,0.2960


In [87]:
# ===============================
# 4. ISSUE EXTRACTION (FINAL)
# ===============================
def extract_issue(text):
    text = str(text).lower()

    issue_map = {
        "battery": ["battery", "drain", "backup", "charging", "charge", "power", "life", "dead"],
        "heating": ["heat", "overheat", "hot", "warm", "temperature"],
        "display": ["display", "screen", "flicker", "touch", "pixel", "crack", "brightness"],
        "performance": ["slow", "lag", "hang", "freeze", "crash", "unresponsive", "delay"],
        "camera": ["camera", "blur", "photo", "focus", "video", "picture"],
        "sound": ["sound", "audio", "speaker", "volume", "mic"],
        "connectivity": ["wifi", "bluetooth", "network", "signal", "disconnect"],
        "build_quality": ["build", "material", "fragile", "scratch", "broken", "cheap"],
        "software": ["software", "update", "bug", "glitch", "app", "ui", "laggy"]
    }

    scores = {}

    for issue, keywords in issue_map.items():
        score = sum(word in text for word in keywords)
        if score > 0:
            scores[issue] = score

    if scores:
        return max(scores, key=scores.get)

    return "none"

df["Issue"] = df["Review"].apply(extract_issue)

# ===============================
# 5. FIX LOW-RATING NONE CASE
# ===============================
def fix_issue(row):
    if row["Issue"] == "none" and row["Rating"] <= 2:
        return "general_issue"
    return row["Issue"]

df["Issue"] = df.apply(fix_issue, axis=1)

# ===============================
# 6. LOWERCASE CONSISTENCY
# ===============================
df["Issue"] = df["Issue"].astype(str).str.lower()

# ===============================
# 7. IMPROVE NONE HANDLING (IMPORTANT FIX)
# ===============================
# If sentiment is strongly negative but no issue → assign general_issue

# Re-create Sentiment_score if it's missing to prevent KeyError
if "Sentiment_score" not in df.columns:
    from nltk.sentiment import SentimentIntensityAnalyzer
    import nltk
    nltk.download('vader_lexicon') # Ensure lexicon is downloaded
    sia = SentimentIntensityAnalyzer()
    df["Sentiment_score"] = df["Review"].apply(lambda x: sia.polarity_scores(str(x))["compound"])

def fix_issue(row):
    if row["Issue"] == "none" and row["Rating"] <= 2 and row["Sentiment"] < -0.2:
        return "general_issue"
    return row["Issue"]

df["Issue"] = df.apply(fix_sentiment_issue, axis=1)

def fix_positive_issue(row):
    if row["Rating"] >=5 and row["Issue"] == "general_issue":
        return "none"
    return row["Issue"]

df["Issue"] = df.apply(fix_positive_issue, axis=1)

In [88]:

# ===============================
# 8. RSI CALCULATION (STABLE)
# ===============================
def calculate_rsi(row):
    sentiment = row["Sentiment_score"]
    rating = row["Rating"]

    neg_sent = max(0, -sentiment)
    rating_dev = abs(5 - rating) / 4
    keyword_intensity = 1 if row["Issue"] != "none" else 0

    # fixed randomness for consistency
    specificity = 0.6

    rsi = (
        0.4 * neg_sent +
        0.3 * rating_dev +
        0.2 * keyword_intensity +
        0.1 * specificity
    )

    return min(max(rsi, 0), 1)

df["RSI"] = df.apply(calculate_rsi, axis=1)

In [89]:
print(df["Issue"].value_counts())
print(df[["Rating", "Sentiment_score", "Issue", "RSI"]].head(10))

Issue
none             3496
general_issue    1447
battery           684
camera            376
performance       365
heating           349
display           323
software          265
build_quality      85
sound              76
connectivity       28
Name: count, dtype: int64
   Rating  Sentiment_score          Issue      RSI
0       5           0.5859           none  0.06000
1       2          -0.4215        display  0.65360
2       5           0.0772           none  0.06000
3       4           0.4404           none  0.13500
4       1           0.2960        heating  0.56000
5       4          -0.5423  general_issue  0.55192
6       2          -0.1695  general_issue  0.55280
7       5           0.7184           none  0.06000
8       5          -0.5423           none  0.27692
9       3          -0.5096  general_issue  0.61384


In [90]:
# ===============================
# 8. FINAL CLEANUP
# ===============================
# No duplicate sentiment column to remove.

# ===============================
# 9. VALIDATION
# ===============================
print("\nFinal Shape:", df.shape)
print("\nIssue Distribution:\n", df["Issue"].value_counts())
print("\nRSI Stats:\n", df["RSI"].describe())

print("\nSample Output:")
print(df[["Rating", "Issue", "RSI"]].head(10)) # 'Sentiment_score' removed to prevent KeyError

# ===============================
# 10. SAVE PROCESSED DATASET
# ===============================
df.to_csv(
    "/content/drive/MyDrive/DWDM_Project/data/processed/samsung_processed_dataset.csv",
    index=False
)

print("\n✅ ETL COMPLETED SUCCESSFULLY")


Final Shape: (7494, 9)

Issue Distribution:
 Issue
none             3496
general_issue    1447
battery           684
camera            376
performance       365
heating           349
display           323
software          265
build_quality      85
sound              76
connectivity       28
Name: count, dtype: int64

RSI Stats:
 count    7494.000000
mean        0.353756
std         0.221033
min         0.060000
25%         0.135000
50%         0.335000
75%         0.551920
max         0.907400
Name: RSI, dtype: float64

Sample Output:
   Rating          Issue      RSI
0       5           none  0.06000
1       2        display  0.65360
2       5           none  0.06000
3       4           none  0.13500
4       1        heating  0.56000
5       4  general_issue  0.55192
6       2  general_issue  0.55280
7       5           none  0.06000
8       5           none  0.27692
9       3  general_issue  0.61384

✅ ETL COMPLETED SUCCESSFULLY
